# Slice-selection ablation

Does CSF-guided slice selection help, and does leaving out the inferior half of the brain (hippocampus, medial temporal lobe) cost performance?
This run cuts six slice sets and scores each with the same frozen-feature classifiers (four ImageNet backbones, logistic regression),
on the same patient-level folds as `patient_r0`:

| Set | Slices |
|---|---|
| `paper` | The paper's own slices (the PNG dataset), as a reference |
| `csf` | CSF-guided selection re-applied to the MNI volumes |
| `ventricle_axial` | Fixed axial 26 mm slab in MNI space through the lateral ventricles, z = 5 to 31 mm |
| `central` | Axial 26 mm slab centred on the middle of the brain mask |
| `mtl_axial` | Fixed axial 26 mm slab in MNI space through the medial temporal lobe, z = −28 to −2 mm |
| `hippo_coronal` | Fixed coronal 26 mm slab in MNI space through the hippocampus, y = −38 to −12 mm |

The five MNI sets come from the same volumes (N4, HD-BET brain extraction, affine registration to MNI152 at 1 mm). Their brain is the
region HD-BET kept, and they go through the paper's own CSF rule and slice processing, copied from `preprocess_nifti.py` (crop to the
brain, the slice's own 1st and 99th percentiles, CLAHE 2.0 with 8×8 tiles, 224×224), turned to the paper's orientation.
Only the volumes and the slab positions differ.

In [ ]:
import os, json, time, warnings, shutil, zipfile, subprocess
import multiprocessing as mp
import numpy as np
from scipy import ndimage
import pandas as pd
from collections import defaultdict
from PIL import Image

import torch
import torch.nn as nn
from torchvision.models import (resnet18, ResNet18_Weights, densenet121, DenseNet121_Weights,
                                resnet50, ResNet50_Weights, efficientnet_b0, EfficientNet_B0_Weights)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import cv2
try:
    import nibabel as nib
except ImportError:
    subprocess.run(['pip', 'install', '-q', 'nibabel'], check=True)
    import nibabel as nib

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


## Configuration

In [ ]:
INPUT_ROOT = '/kaggle/input'
WORK_DIR   = '/kaggle/working'
OUT_DIR    = os.path.join(WORK_DIR, 'results_ablation_quick')
os.makedirs(OUT_DIR, exist_ok=True)

PNG_ROOT = None
for root, dirs, files in os.walk(INPUT_ROOT):
    if os.path.basename(root) == 'adni_nifti_preprocessed' and {'AD', 'CN', 'MCI'} <= set(dirs):
        PNG_ROOT = root
        break
assert PNG_ROOT, 'Attach the ADNI slice dataset (adni_nifti_preprocessed with AD, CN and MCI)'

DRIVE_FILE_ID = '1JScVT0E9QVSOB5a8s8CciWixY7rs2vx8'
ZIP_NAME      = 'nifti_preprocessed.zip'
ZIP_BYTES     = 8036136248
DOWNLOAD_DIRS = ['/tmp', WORK_DIR]

SEED, CV_SEED, N_OUTER, CAL_FRACTION = 42, 100, 5, 0.20   # the folds of patient_r0
IMAGE_SIZE = 224
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
SLAB_MM = 26.0
VENTRICLE_Z_MM = 18.0       # centre of z = 5 to 31 mm
MTL_AXIAL_Z_MM = -15.0      # centre of z = -28 to -2 mm
HIPPO_CORONAL_Y_MM = -25.0  # centre of y = -38 to -12 mm
CLAHE_CLIP, CLAHE_GRID = 2.0, (8, 8)
PRETRAINED = True
BACKBONES = ['resnet18+densenet121', 'resnet50', 'densenet121', 'efficientnet_b0']
PCA_DIM, N_BOOT = 256, 2000
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
CLASS_NAMES = ['AD', 'CN', 'MCI']
SETS = ['paper', 'csf', 'ventricle_axial', 'central', 'mtl_axial', 'hippo_coronal']
MNI_SETS = SETS[1:]
np.random.seed(SEED)
torch.manual_seed(SEED)
print('slices:', PNG_ROOT)

slices: /kaggle/input/datasets/malakkhaled202300369/adni-data/adni_nifti_preprocessed


## Download the MNI volumes

In [ ]:
ZIP_PATH = None
for d in DOWNLOAD_DIRS:
    p = os.path.join(d, ZIP_NAME)
    if os.path.exists(p) and os.path.getsize(p) == ZIP_BYTES:
        ZIP_PATH = p
        break

if ZIP_PATH is None:
    need = ZIP_BYTES + 3 * 10**9
    free = {d: shutil.disk_usage(d).free for d in DOWNLOAD_DIRS}
    print('Free disk (GB):', {d: round(v / 1e9, 1) for d, v in free.items()})
    target = next((d for d in DOWNLOAD_DIRS if free[d] > need), None)
    assert target, f'Not enough free disk: about {need / 1e9:.0f} GB is needed'
    ZIP_PATH = os.path.join(target, ZIP_NAME)
    if os.path.exists(ZIP_PATH):
        os.remove(ZIP_PATH)
    subprocess.run(['pip', 'install', '-q', '-U', 'gdown'], check=True)
    import gdown, inspect
    can_resume = 'resume' in inspect.signature(gdown.download).parameters
    extra = {'resume': True} if can_resume else {}
    t0, last_error = time.time(), None
    for attempt in range(5):
        try:
            gdown.download(id=DRIVE_FILE_ID, output=ZIP_PATH, quiet=False, **extra)
        except Exception as e:
            last_error = e
        size = os.path.getsize(ZIP_PATH) if os.path.exists(ZIP_PATH) else 0
        if size == ZIP_BYTES:
            break
        if os.path.exists(ZIP_PATH):
            if can_resume:
                os.replace(ZIP_PATH, ZIP_PATH + '.part')
            else:
                os.remove(ZIP_PATH)
        print(f'Attempt {attempt + 1} stopped before the end, retrying')
    size = os.path.getsize(ZIP_PATH) if os.path.exists(ZIP_PATH) else 0
    if size != ZIP_BYTES:
        raise RuntimeError('Download failed. '
                           f'Last error: {last_error}')
    print(f'Downloaded in {(time.time() - t0) / 60:.1f} min')

zf = zipfile.ZipFile(ZIP_PATH)
members = [n for n in zf.namelist() if n.lower().endswith('.nii.gz') or n.lower().endswith('.nii')]
assert members, 'The zip contains no .nii or .nii.gz files'
print(f'{ZIP_PATH}: {len(members)} NIfTI files, for example {members[0]}')

Free disk (GB): {'/tmp': 1038.2, '/kaggle/working': 20.9}


Downloading...
From (original): https://drive.google.com/uc?id=1JScVT0E9QVSOB5a8s8CciWixY7rs2vx8
From (redirected): https://drive.google.com/uc?id=1JScVT0E9QVSOB5a8s8CciWixY7rs2vx8&confirm=t&uuid=3ef1d120-986c-4bf9-aa4a-752459c0b0a9
To: /tmp/nifti_preprocessed.zip
100%|██████████| 8.04G/8.04G [03:30<00:00, 38.1MB/s]


Downloaded in 3.6 min
/tmp/nifti_preprocessed.zip: 389 NIfTI files, for example nifti_preprocessed/002_S_1155.nii.gz


## Cohort and the folds of patient_r0

In [ ]:
patient_classes = defaultdict(set)
patient_files = {}
for cls in CLASS_NAMES:
    cls_dir = os.path.join(PNG_ROOT, cls)
    for pid in sorted(os.listdir(cls_dir)):
        pdir = os.path.join(cls_dir, pid)
        if not os.path.isdir(pdir):
            continue
        files = sorted(os.path.join(pdir, f) for f in os.listdir(pdir) if os.path.splitext(f)[1].lower() in VALID_EXTS)
        if files:
            patient_classes[pid].add(cls)
            patient_files[(pid, cls)] = files
patients = sorted(p for p, c in patient_classes.items() if len(c) == 1)
patient_label = {p: CLASS_NAMES.index(next(iter(patient_classes[p]))) for p in patients}

y_pat = np.array([patient_label[p] for p in patients])
folds = []
skf = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=CV_SEED)
for k, (tr, te) in enumerate(skf.split(np.zeros(len(patients)), y_pat)):
    tr_p = [patients[i] for i in tr]
    te_p = [patients[i] for i in te]
    fit_p, cal_p = train_test_split(tr_p, test_size=CAL_FRACTION, stratify=[patient_label[p] for p in tr_p],
                                    random_state=1000 + k)
    folds.append({'fit': fit_p, 'cal': cal_p, 'test': te_p})

# match each patient to one volume in the zip: a file named <PID>.nii(.gz), or the only NIfTI file inside a folder named <PID>
def stem(name):
    b = name.replace('\\', '/').split('/')[-1]
    return b[:-7] if b.lower().endswith('.nii.gz') else b[:-4]

by_stem = defaultdict(list)
for n in members:
    by_stem[stem(n)].append(n)
vol_member, ambiguous = {}, {}
for p in patients:
    cands = by_stem.get(p) or [n for n in members if p in n.replace('\\', '/').split('/')[:-1]]
    if len(cands) == 1:
        vol_member[p] = cands[0]
    elif len(cands) > 1:
        ambiguous[p] = cands
assert not ambiguous, f'{len(ambiguous)} patients match more than one volume, for example: {next(iter(ambiguous.values()))[:4]}'
missing = [p for p in patients if p not in vol_member]
print(f'Patients: {len(patients)} | with a volume: {len(vol_member)} | without a volume: {len(missing)}', missing[:10])
assert vol_member, 'No patient ID matched a file in the zip; compare the example file name printed above with the patient folders'

Patients: 320 | with a volume: 320 | without a volume: 0 []


## Slice sets

**Brain mask.** The MNI volumes were brain-extracted (HD-BET) before registration: outside the brain they are zero apart from
interpolation noise of about 1e-8. The brain is every voxel above 0.1% of the brain's 99th-percentile intensity (largest connected
component, holes filled), which keeps the ventricles, the sulci and the cortex. A volume with more than 3.5 l above that level still
contains the head and goes through the paper's own 3D extraction from `preprocess_nifti.py` instead. Volumes whose final mask is not
1.2 to 3.5 l are left out of the MNI sets and counted.

**Front-back check.** In MNI space the cerebellum and brainstem make the posterior half of the brain reach about 30 mm lower than the
anterior half. For each volume the lowest brain voxel of each half is compared; if the half the header calls anterior reaches lower,
the volume's front-back and left-right axes are reversed with respect to its header (a common effect of saving an ITK array with an
MNI header), and both axes are flipped back before any slab is cut. Volumes where the two halves differ by 10 mm or less are kept as
they are and counted.

**Slabs.** The CSF rule, the 13 mm on each side and the per-slice crop, percentile normalisation and CLAHE are copied from
our originalpreprocessing notebook. The fixed slabs are placed in MNI millimetres.

**Orientation.** The axial slices are rotated or flipped to match the paper's PNG slices: the notebook tries the 8 rotations and
flips, compares the average ventricle-slab slice with the average paper slice of the same patients, and keeps the best match.
Coronal slices are turned so that superior is at the top, with left and right on the same sides as in the axial slices.

In [ ]:
def extract_brain_mask_3d(vol_arr):
    nonzero = vol_arr[vol_arr > 0]
    if len(nonzero) == 0:
        return np.zeros_like(vol_arr, dtype=bool)
    p1, p99 = np.percentile(nonzero, (1.0, 99.0))
    clipped = np.clip(vol_arr, p1, p99)
    norm_u8 = ((clipped - p1) / (p99 - p1 + 1e-8) * 255).astype(np.uint8)
    thresh_val, _ = cv2.threshold(norm_u8.flatten(), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    binary_mask = norm_u8 > (thresh_val * 0.75)
    labeled, num_features = ndimage.label(binary_mask)
    if num_features == 0:
        return binary_mask
    sizes = ndimage.sum(binary_mask, labeled, range(1, num_features + 1))
    brain_core = labeled == (np.argmax(sizes) + 1)
    brain_core = ndimage.binary_fill_holes(brain_core)
    dist = ndimage.distance_transform_edt(brain_core)
    core = dist > 7.0
    if np.any(core):
        core_labeled, core_num = ndimage.label(core)
        core_sizes = ndimage.sum(core, core_labeled, range(1, core_num + 1))
        best_core = core_labeled == (np.argmax(core_sizes) + 1)
        dilated = ndimage.binary_dilation(best_core, iterations=7)
        return dilated & brain_core
    return brain_core

def locate_ventricle_center(brain_arr, mask_arr):
    z_indices = np.where(np.any(mask_arr, axis=(0, 1)))[0]
    if len(z_indices) == 0:
        return brain_arr.shape[2] // 2
    brain_start, brain_end = z_indices[0], z_indices[-1]
    brain_depth = brain_end - brain_start
    search_start = brain_start + int(brain_depth * 0.20)
    search_end = brain_start + int(brain_depth * 0.50)
    brain_voxels = brain_arr[mask_arr]
    med_val = float(np.median(brain_voxels)) if len(brain_voxels) > 0 else 1.0
    best_score, best_z = -1.0, (search_start + search_end) // 2
    for z in range(search_start, search_end + 1):
        sl_brain, sl_mask = brain_arr[:, :, z], mask_arr[:, :, z]
        if len(sl_brain[sl_mask]) < 100:
            continue
        csf_volume = int(np.sum(sl_mask & (sl_brain < (med_val * 0.55)) & (sl_brain > 0)))
        if csf_volume > best_score:
            best_score, best_z = csf_volume, z
    return best_z

def process_slice(sl, clahe):
    # skip slices under 5% brain, tight crop (margin 4), the slice's own 1st/99th percentiles, CLAHE, 224x224
    if np.mean(sl > 0) < 0.05:
        return None
    coords = np.argwhere(sl > 0)
    y0, x0 = np.maximum(coords.min(0) - 4, 0)
    y1, x1 = np.minimum(coords.max(0) + 4, np.array(sl.shape) - 1)
    crop = sl[y0:y1 + 1, x0:x1 + 1]
    vals = crop[crop > 0]
    if len(vals) == 0:
        return None
    p1, p99 = np.percentile(vals, (1.0, 99.0))
    u8 = (np.clip((crop - p1) / (p99 - p1 + 1e-8), 0.0, 1.0) * 255).astype(np.uint8)
    return cv2.resize(clahe.apply(u8), (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)

STRIPPED_MAX_ML = 3500.0          # above this, the volume still contains the head
MASK_OK_ML = (1200.0, 3500.0)     # plausible brain masks after affine registration to MNI152 (HD-BET keeps the CSF around the brain)
AP_MIN_MM = 10.0                  # smallest difference between the two halves' lowest points that counts as a clear front-back answer

def brain_mask_for(vol, voxel_ml):
    pos = vol[vol > 0]
    if len(pos) == 0:
        return np.zeros(vol.shape, dtype=bool), 0.0, 'empty'
    support = vol > 1e-3 * float(np.percentile(pos, 99))
    support_ml = float(support.sum() * voxel_ml)
    if support_ml >= STRIPPED_MAX_ML:
        return extract_brain_mask_3d(vol), support_ml, 'paper extraction'
    lab, n = ndimage.label(support)
    if n == 0:
        return support, support_ml, 'empty'
    sizes = ndimage.sum(support, lab, range(1, n + 1))
    return ndimage.binary_fill_holes(lab == (np.argmax(sizes) + 1)), support_ml, 'already brain-extracted'

def front_back_difference(mask, dz):
    # RAS+ array: axis 1 grows towards the header's anterior, axis 2 towards superior.
    # Returns how much lower (mm) the header-posterior half reaches than the header-anterior half; about +30 when the header is right.
    jk = mask.any(axis=0)
    js = np.where(jk.any(axis=1))[0]
    jmid = (js[0] + js[-1]) // 2
    low_post = np.where(jk[:jmid].any(axis=0))[0].min()
    low_ant = np.where(jk[jmid:].any(axis=0))[0].min()
    return float((low_ant - low_post) * dz)

def index_for_mm(affine, axis, mm):
    return int(round((mm - affine[axis, 3]) / affine[axis, axis]))

def slab(brain, axis, centre, spacing, clahe):
    half = max(3, int((SLAB_MM / 2.0) / spacing))
    out = []
    for i in range(max(0, centre - half), min(brain.shape[axis] - 1, centre + half) + 1):
        img = process_slice(brain[:, :, i] if axis == 2 else brain[:, i, :], clahe)
        if img is not None:
            out.append(img)
    return out

def init_worker(zip_path, tmp_dir):
    global W_ZF, W_TMP, W_CLAHE
    W_ZF = zipfile.ZipFile(zip_path)
    W_TMP = tmp_dir
    W_CLAHE = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_GRID)

def process_patient(job):
    p, member = job
    path = os.path.join(W_TMP, f'vol_{os.getpid()}' + ('.nii.gz' if member.lower().endswith('.gz') else '.nii'))
    with W_ZF.open(member) as src, open(path, 'wb') as dst:
        shutil.copyfileobj(src, dst)
    img = nib.as_closest_canonical(nib.load(path))   # RAS+
    vol = np.asarray(img.dataobj, dtype=np.float32)
    affine = np.array(img.affine)
    os.remove(path)
    voxel_ml = float(np.prod(np.abs(np.diag(affine)[:3]))) / 1000.0
    diag = {'min': float(vol.min()), 'zero_share': float(np.mean(vol == 0)), 'shape': tuple(vol.shape),
            'spacing': tuple(float(v) for v in np.round(np.abs(np.diag(affine)[:3]), 2)),
            'off_diag': float(np.abs(affine[:3, :3] - np.diag(np.diag(affine[:3, :3]))).max())}
    mask, support_ml, how = brain_mask_for(vol, voxel_ml)
    mask_ml = float(mask.sum() * voxel_ml)
    ok = MASK_OK_ML[0] <= mask_ml <= MASK_OK_ML[1]
    diag.update({'support_ml': support_ml, 'mask_ml': mask_ml, 'how': how, 'ok': ok})
    if not ok:
        return p, None, None, diag
    # front-back check, and the fix for volumes stored reversed with respect to their header
    ap = front_back_difference(mask, abs(affine[2, 2]))
    diag['ap_mm'] = ap
    if ap < -AP_MIN_MM:
        vol, mask = vol[::-1, ::-1, :], mask[::-1, ::-1, :]
        diag['ap'] = 'reversed, flipped back'
    else:
        diag['ap'] = 'as in header' if ap > AP_MIN_MM else 'unclear, kept'
    # the paper's array order (ANTs "RAS"): index 0 is right, anterior and superior
    vol = np.ascontiguousarray(vol[::-1, ::-1, ::-1])
    mask = np.ascontiguousarray(mask[::-1, ::-1, ::-1])
    brain = np.where(mask, np.clip(vol, 0, None), 0).astype(np.float32)
    nx, ny, nz = vol.shape
    to_mm = lambda axis, i, n: float(affine[axis, axis] * (n - 1 - i) + affine[axis, 3])   # paper index to MNI mm
    from_mm = lambda axis, mm, n: n - 1 - index_for_mm(affine, axis, mm)                    # MNI mm to paper index
    zs = np.where(mask.any(axis=(0, 1)))[0]
    z_csf = int(locate_ventricle_center(brain, mask))
    z_mid = int((zs[0] + zs[-1]) // 2)
    pos = {'csf': (2, z_csf), 'ventricle_axial': (2, from_mm(2, VENTRICLE_Z_MM, nz)), 'central': (2, z_mid),
           'mtl_axial': (2, from_mm(2, MTL_AXIAL_Z_MM, nz)), 'hippo_coronal': (1, from_mm(1, HIPPO_CORONAL_Y_MM, ny))}
    spacing = {1: abs(affine[1, 1]), 2: abs(affine[2, 2])}
    slabs = {s: slab(brain, axis, i, spacing[axis], W_CLAHE) for s, (axis, i) in pos.items()}
    centres_mm = {s: to_mm(axis, i, nz if axis == 2 else ny) for s, (axis, i) in pos.items()}
    diag.update({'brain_top_mm': to_mm(2, int(zs[0]), nz), 'brain_bottom_mm': to_mm(2, int(zs[-1]), nz)})
    return p, slabs, centres_mm, diag

data = {s: {'img': [], 'pid': []} for s in SETS}
centres = {s: [] for s in MNI_SETS}
diags, mid_ven = [], {}
t0 = time.time()
for p in patients:
    for f in patient_files[(p, CLASS_NAMES[patient_label[p]])]:
        data['paper']['img'].append(np.array(Image.open(f).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)))
        data['paper']['pid'].append(p)
jobs = [(p, vol_member[p]) for p in patients if p in vol_member]
n_workers = max(1, min(4, os.cpu_count() or 1))
with mp.get_context('fork').Pool(n_workers, initializer=init_worker, initargs=(ZIP_PATH, os.path.dirname(ZIP_PATH))) as pool:
    for p, slabs, cmm, diag in pool.imap(process_patient, jobs, chunksize=2):
        diags.append(diag)
        if slabs is None:
            continue
        if slabs['ventricle_axial']:
            mid_ven[p] = slabs['ventricle_axial'][len(slabs['ventricle_axial']) // 2]
        for s in MNI_SETS:
            centres[s].append(cmm[s])
            data[s]['img'].extend(slabs[s])
            data[s]['pid'].extend([p] * len(slabs[s]))
print(f'{len(jobs)} volumes processed with {n_workers} workers in {(time.time() - t0) / 60:.1f} min')

# what the volumes look like
checks = {}
dg = pd.DataFrame(diags)
good = dg[dg['ok']]
print('Volume shapes:', set(dg['shape']), '| voxel sizes:', set(dg['spacing']), '| largest off-diagonal affine term:', round(dg['off_diag'].max(), 4))
print(f"Voxels exactly 0: median {dg['zero_share'].median() * 100:.1f}% | lowest value: median {dg['min'].median():.3g}")
print('Brain mask from:', dg['how'].value_counts().to_dict())
print(f"Voxels above 0.1% of the brain intensity: {dg['support_ml'].median():.0f} ml median "
      f"({dg['support_ml'].quantile(0.05):.0f} to {dg['support_ml'].quantile(0.95):.0f}, 5th to 95th pct)")
print(f"Brain mask: {good['mask_ml'].median():.0f} ml median ({good['mask_ml'].quantile(0.05):.0f} to {good['mask_ml'].quantile(0.95):.0f}) | "
      f"left out (mask outside 1.2 to 3.5 l): {int((~dg['ok']).sum())} of {len(dg)} | "
      f"brain top z = {good['brain_top_mm'].median():.0f} mm, bottom z = {good['brain_bottom_mm'].median():.0f} mm (medians)")
print('Front-back check:', good['ap'].value_counts().to_dict(),
      f"| posterior half reaches lower by {good['ap_mm'].median():.0f} mm (median, as stored)")
for s in MNI_SETS:
    c = np.array(centres[s])
    print(f"{s}: slab centre {'y' if s == 'hippo_coronal' else 'z'} = {c.mean():.1f} ± {c.std():.1f} mm "
          f"(5th to 95th pct {np.percentile(c, 5):.0f} to {np.percentile(c, 95):.0f})")
csf_c = np.array(centres['csf'])
near = float(np.mean((csf_c >= 5) & (csf_c <= 31)))
print(f"CSF rule on these volumes: slab centre within the ventricle range (z 5 to 31 mm) for {near * 100:.0f}% of patients")
checks.update({'n_volumes': int(len(dg)), 'left_out': int((~dg['ok']).sum()), 'mask_from': dg['how'].value_counts().to_dict(),
               'mask_ml_median': float(good['mask_ml'].median()), 'front_back': good['ap'].value_counts().to_dict(),
               'front_back_mm_median': float(good['ap_mm'].median()), 'csf_centre_in_ventricle_range': near})

assert dg['ok'].mean() >= 0.9, 'More than 10% of the brain masks are implausible; see the numbers printed above'
assert np.std(centres['central']) > 0, 'The central slab is at the same position for every patient, so the brain mask is not working'

# orientation: match the axial MNI slices to the paper's PNG slices of the same patients
TRANSFORMS = {'as is': lambda a: a, 'rot90': lambda a: np.rot90(a, 1), 'rot180': lambda a: np.rot90(a, 2),
              'rot270': lambda a: np.rot90(a, 3), 'flip': lambda a: a[:, ::-1], 'flip, rot90': lambda a: np.rot90(a[:, ::-1], 1),
              'flip, rot180': lambda a: np.rot90(a[:, ::-1], 2), 'flip, rot270': lambda a: np.rot90(a[:, ::-1], 3)}
pairs = [p for p in patients if p in mid_ven]
paper_mid = {}
for p in pairs:
    files = patient_files[(p, CLASS_NAMES[patient_label[p]])]
    paper_mid[p] = np.array(Image.open(files[len(files) // 2]).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS), dtype=np.float32)
corr = lambda a, b: float(np.corrcoef(a.ravel(), b.ravel())[0, 1])
mean_paper = np.mean([paper_mid[p] for p in pairs], axis=0)
mean_mni = np.mean([mid_ven[p].astype(np.float32) for p in pairs], axis=0)
scores = {name: corr(mean_paper, f(mean_mni)) for name, f in TRANSFORMS.items()}
best = max(scores, key=scores.get)
per_patient = [max(TRANSFORMS, key=lambda n: corr(paper_mid[p], TRANSFORMS[n](mid_ven[p].astype(np.float32)))) for p in pairs]
mirror = {'as is': 'flip', 'flip': 'as is', 'rot90': 'flip, rot270', 'flip, rot270': 'rot90',
          'rot180': 'flip, rot180', 'flip, rot180': 'rot180', 'rot270': 'flip, rot90', 'flip, rot90': 'rot270'}
agree = float(np.mean([t == best for t in per_patient]))
agree_lr = float(np.mean([t in (best, mirror[best]) for t in per_patient]))
print('Orientation match (correlation of the average slices):', {k: round(v, 3) for k, v in sorted(scores.items(), key=lambda kv: -kv[1])})
print(f"Chosen: {best} | patients whose own best match agrees: {agree * 100:.0f}% ({agree_lr * 100:.0f}% allowing a left-right flip)")
T_axial = TRANSFORMS[best]

# coronal slices (rows: left-right, columns: superior to inferior): superior at the top, left and right as in the axial slices
probe = np.tile(np.arange(IMAGE_SIZE, dtype=np.float32)[:, None], (1, IMAGE_SIZE))   # value = left-right index
tp = T_axial(probe)
x_horizontal = abs(tp[0, -1] - tp[0, 0]) > abs(tp[-1, 0] - tp[0, 0])
same_direction = (not x_horizontal) or (tp[0, -1] > tp[0, 0])
T_coronal = (lambda a: a.T) if same_direction else (lambda a: a.T[:, ::-1])
checks.update({'orientation': best, 'orientation_scores': scores, 'orientation_agreement': agree, 'orientation_agreement_lr': agree_lr})

for s in MNI_SETS:
    T = T_coronal if s == 'hippo_coronal' else T_axial
    data[s]['img'] = [np.ascontiguousarray(T(im)) for im in data[s]['img']]

for s in SETS:
    data[s]['img'] = np.stack(data[s]['img'])
    data[s]['pid'] = np.array(data[s]['pid'])
    data[s]['label'] = np.array([patient_label[p] for p in data[s]['pid']])
    idx = defaultdict(list)
    for i, p in enumerate(data[s]['pid']):
        idx[p].append(i)
    data[s]['idx'] = {p: np.array(v) for p, v in idx.items()}
    pick = [p for c in range(3) for p in [q for q in patients if patient_label[q] == c and q in data[s]['idx']][:2]]
    tiles = [data[s]['img'][data[s]['idx'][p][len(data[s]['idx'][p]) // 2]] for p in pick]
    Image.fromarray(np.concatenate(tiles, axis=1)).save(os.path.join(OUT_DIR, f'qc_{s}.png'))
    per = np.array([len(v) for v in data[s]['idx'].values()])
    print(f"{s}: {len(data[s]['idx'])} patients, {per.mean():.1f} slices per patient")

320 volumes processed with 4 workers in 13.5 min
Volume shapes: {(193, 229, 193)} | voxel sizes: {(1.0, 1.0, 1.0)} | largest off-diagonal affine term: 0.0
Voxels exactly 0: median 23.1% | lowest value: median -2.19e-08
Brain mask from: {'already brain-extracted': 320}
Voxels above 0.1% of the brain intensity: 2088 ml median (1902 to 2794, 5th to 95th pct)
Brain mask: 2100 ml median (1915 to 2811) | left out (mask outside 1.2 to 3.5 l): 0 of 320 | brain top z = 85 mm, bottom z = -74 mm (medians)
Front-back check: {'as in header': 315, 'unclear, kept': 5} | posterior half reaches lower by 20 mm (median, as stored)
csf: slab centre z = 35.5 ± 23.2 mm (5th to 95th pct 3 to 76)
ventricle_axial: slab centre z = 18.0 ± 0.0 mm (5th to 95th pct 18 to 18)
central: slab centre z = 7.7 ± 4.5 mm (5th to 95th pct 3 to 18)
mtl_axial: slab centre z = -15.0 ± 0.0 mm (5th to 95th pct -15 to -15)
hippo_coronal: slab centre y = -25.0 ± 0.0 mm (5th to 95th pct -25 to -25)
CSF rule on these volumes: slab ce

## Frozen ImageNet features for every set

In [ ]:
def build_extractor(name):
    if name == 'resnet18+densenet121':
        r = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        r.fc = nn.Identity()
        d = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        d.classifier = nn.Identity()
        class Both(nn.Module):
            def __init__(self):
                super().__init__()
                self.r, self.d = r, d
            def forward(self, x):
                return torch.cat([self.r(x), self.d(x)], dim=1)
        return Both()
    if name == 'resnet50':
        m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if PRETRAINED else None)
        m.fc = nn.Identity()
        return m
    if name == 'densenet121':
        m = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        m.classifier = nn.Identity()
        return m
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if PRETRAINED else None)
    m.classifier = nn.Identity()
    return m

t0 = time.time()
feats = {s: {} for s in SETS}
for name in BACKBONES:
    ext = build_extractor(name).to(device).eval()
    for s in SETS:
        imgs, chunks = data[s]['img'], []
        with torch.no_grad():
            for i in range(0, len(imgs), 128):
                x = torch.from_numpy(imgs[i:i + 128]).float().div(255.0).unsqueeze(1).repeat(1, 3, 1, 1)
                x = ((x - MEAN) / STD).to(device)
                with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                    chunks.append(ext(x).float().cpu().numpy())
        feats[s][name] = np.concatenate(chunks)
    del ext
    if device.type == 'cuda':
        torch.cuda.empty_cache()
print(f'Features done in {(time.time() - t0) / 60:.1f} min')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 212MB/s]


Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 102MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 245MB/s]


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 209MB/s]


Features done in 5.6 min


## Logistic regression per set, backbone chosen on the calibration patients

In [ ]:
def metrics3(y, pred):
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1, 2], zero_division=0)
    return {'acc': float(accuracy_score(y, pred)), 'macro_P': float(p.mean()), 'macro_R': float(r.mean()),
            'macro_F1': float(f.mean()), 'R_AD': float(r[0]), 'R_CN': float(r[1]), 'R_MCI': float(r[2])}

def boot_ci(y, pred):
    y, pred = np.asarray(y), np.asarray(pred)
    rng = np.random.default_rng(SEED)
    vals = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        vals.append(precision_recall_fscore_support(y[i], pred[i], labels=[0, 1, 2], zero_division=0)[2].mean())
    return [float(v) for v in np.percentile(vals, [2.5, 97.5])]

def patient_mean(prob, pids, order):
    df = pd.DataFrame(prob)
    df['pid'] = pids
    return df.groupby('pid').mean().loc[order].values

results = {}
t0 = time.time()
for s in SETS:
    d = data[s]
    have = lambda ps: [p for p in ps if p in d['idx']]
    ix = lambda ps: np.concatenate([d['idx'][p] for p in ps])
    rows, slice_true, slice_pred, chosen = [], [], defaultdict(list), []
    for k, f in enumerate(folds):
        fit_p, cal_p, te_p = have(f['fit']), have(f['cal']), have(f['test'])
        fit_i, cal_i, te_i = ix(fit_p), ix(cal_p), ix(te_p)
        y_cal = np.array([patient_label[p] for p in cal_p])
        te_prob, cal_f1 = {}, {}
        for bb in BACKBONES:
            clf = make_pipeline(StandardScaler(), PCA(n_components=min(PCA_DIM, len(fit_i) - 1), random_state=SEED),
                                LogisticRegression(C=0.1, class_weight='balanced', max_iter=3000))
            clf.fit(feats[s][bb][fit_i], d['label'][fit_i])
            cal_pat = patient_mean(clf.predict_proba(feats[s][bb][cal_i]), d['pid'][cal_i], cal_p)
            cal_f1[bb] = metrics3(y_cal, cal_pat.argmax(1))['macro_F1']
            prob = clf.predict_proba(feats[s][bb][te_i])
            slice_pred[bb].append(prob.argmax(1))
            te_prob[bb] = patient_mean(prob, d['pid'][te_i], te_p)
        slice_true.append(d['label'][te_i])
        bb_sel = max(BACKBONES, key=lambda b: cal_f1[b])
        chosen.append(bb_sel)
        for i, p in enumerate(te_p):
            row = {'patient': p, 'true': patient_label[p], 'selected': int(te_prob[bb_sel][i].argmax())}
            for bb in BACKBONES:
                row[bb] = int(te_prob[bb][i].argmax())
            rows.append(row)
    df = pd.DataFrame(rows)
    st = np.concatenate(slice_true)
    res = {'n_patients': int(len(df)), 'slices_per_patient': float(np.mean([len(v) for v in d['idx'].values()])),
           'chosen_backbones': pd.Series(chosen).value_counts().to_dict(), 'patient': {}, 'slice': {}}
    if s in centres:
        res['slab_centre_mm'] = [float(np.mean(centres[s])), float(np.std(centres[s]))]
    for name in ['selected'] + BACKBONES:
        m = metrics3(df['true'].values, df[name].values)
        m['macro_F1_CI'] = boot_ci(df['true'].values, df[name].values)
        res['patient'][name] = m
    for bb in BACKBONES:
        pr = np.concatenate(slice_pred[bb])
        res['slice'][bb] = float(precision_recall_fscore_support(st, pr, labels=[0, 1, 2], zero_division=0)[2].mean())
    results[s] = res
print(f'Classifiers done in {(time.time() - t0) / 60:.1f} min')

Classifiers done in 11.7 min


## Table

In [ ]:
names = {'paper': "Paper's slices (CSF-guided, original preprocessing)", 'csf': 'CSF-guided, MNI volumes',
         'ventricle_axial': 'Axial slab, lateral ventricles (MNI z 5 to 31 mm)',
         'central': 'Central axial slab', 'mtl_axial': 'Axial slab, medial temporal lobe (MNI z -28 to -2 mm)',
         'hippo_coronal': 'Coronal slab, hippocampus (MNI y -38 to -12 mm)'}
rows = []
for s in SETS:
    r = results[s]
    sel = r['patient']['selected']
    row = {'slice set': names[s], 'patients': r['n_patients'], 'slices/patient': round(r['slices_per_patient'], 1),
           'macro-F1 (backbone chosen on cal)': f"{sel['macro_F1']:.3f} [{sel['macro_F1_CI'][0]:.2f}, {sel['macro_F1_CI'][1]:.2f}]",
           'BA': f"{sel['macro_R']:.3f}", 'recall AD/CN/MCI': f"{sel['R_AD']:.2f}/{sel['R_CN']:.2f}/{sel['R_MCI']:.2f}"}
    for bb in BACKBONES:
        row[bb + ' macro-F1'] = f"{r['patient'][bb]['macro_F1']:.3f}"
    if 'slab_centre_mm' in r:
        row['slab centre (mm)'] = f"{r['slab_centre_mm'][0]:.1f} ± {r['slab_centre_mm'][1]:.1f}"
    rows.append(row)
table = pd.DataFrame(rows).fillna('')
print(table.to_string(index=False))

cols = list(table.columns)
lines = ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
for _, r in table.iterrows():
    lines.append('| ' + ' | '.join(str(r[c]) for c in cols) + ' |')
with open(os.path.join(OUT_DIR, 'ablation_tables.md'), 'w') as fh:
    fh.write('\n'.join(lines))
with open(os.path.join(OUT_DIR, 'ablation_summary.json'), 'w') as fh:
    json.dump(dict(results, checks=checks), fh, indent=1, default=float)
print('Saved ablation_tables.md, ablation_summary.json and the qc_*.png images in', OUT_DIR)

zf.close()
if os.path.abspath(ZIP_PATH).startswith(os.path.abspath(WORK_DIR)):
    os.remove(ZIP_PATH)
    print('Removed the zip from the working folder so it is not saved as output')

                                            slice set  patients  slices/patient macro-F1 (backbone chosen on cal)    BA recall AD/CN/MCI resnet18+densenet121 macro-F1 resnet50 macro-F1 densenet121 macro-F1 efficientnet_b0 macro-F1 slab centre (mm)
  Paper's slices (CSF-guided, original preprocessing)       320            26.6                0.411 [0.35, 0.47] 0.415   0.24/0.45/0.55                         0.435             0.381                0.445                    0.436                 
                              CSF-guided, MNI volumes       320            27.0                0.357 [0.30, 0.41] 0.358   0.20/0.34/0.53                         0.365             0.372                0.372                    0.326      35.5 ± 23.2
    Axial slab, lateral ventricles (MNI z 5 to 31 mm)       320            27.0                0.326 [0.28, 0.38] 0.327   0.11/0.31/0.56                         0.379             0.342                0.366                    0.316       18.0 ± 0.0
        